In [ ]:
MODEL_LABELS = {
    "deepseek_7b": "DeepSeek", "mistral_7b": "Mistral", "phi35_mini": "Phi-3.5",
    "gemma3_4b": "Gemma-3", "llama31_8b": "Llama-3.1", "qwen2_7b": "Qwen2",
    "olmo2_7b": "OLMo-2",
}

In [ ]:
import json, re

def extract_json_object(text):
    if not isinstance(text, str):
        return None
    start = text.find("{")
    if start == -1:
        return None
    depth, in_string, escape = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:      escape = False
            elif ch == "\\": escape = True
            elif ch == '"':  in_string = False
            continue
        if ch == '"':   in_string = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def robust_parse(text):
    cand = extract_json_object(text)
    if cand is None:
        return None
    try:
        return json.loads(cand)
    except json.JSONDecodeError:
        repaired = re.sub(r'(:\s*\d+),(\d+)(?=\s*[,}])', r'\1.\2', cand)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            return None

def extract_adjectives(raw_text):
    data = robust_parse(raw_text)
    if data is None or "error_parsing" in data:
        return pd.Series([np.nan]*8, index=ADJECTIVES)
    return pd.Series(
        [pd.to_numeric(data.get(a, np.nan), errors="coerce") for a in ADJECTIVES],
        index=ADJECTIVES
    )

In [ ]:
import pandas as pd
import numpy as np
import pingouin as pg
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 180); pd.set_option("display.max_columns", None)

ADJECTIVES = ["tolerant","warm","good-natured","sincere",
              "competent","confident","independent","intelligent"]
ETA_SMALL, ETA_MEDIUM = 0.01, 0.06

def _col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"nessuna di {names} in {df.columns.tolist()}")



def coerce(df):
    d = df.copy()
    for a in ADJECTIVES:
        d[a] = pd.to_numeric(d[a], errors="coerce")
    return d

def block_a_vs_baseline(df):
    rows = []
    ethnicities = [e for e in df["ethnicity"].unique() if e != "baseline"]
    for model in df["model_id"].unique():
        dm = df[df["model_id"] == model]
        for adj in ADJECTIVES:
            base_vals = dm[dm["ethnicity"] == "baseline"][adj].dropna()
            if len(base_vals) < 10:
                continue
            for eth in ethnicities:
                eth_vals = dm[dm["ethnicity"] == eth][adj].dropna()
                if len(eth_vals) < 10 or eth_vals.nunique() < 2:
                    continue
                res = pg.ttest(eth_vals, base_vals, correction=True)
                
                delta = eth_vals.mean() - base_vals.mean()
                
                signed_g = pg.compute_effsize(eth_vals, base_vals, eftype="hedges")  
                
                rows.append({
                    "model_id": model, "adjective": adj, "ethnicity": eth,
                    "mean_baseline": base_vals.mean(), "mean_ethnicity": eth_vals.mean(),
                    "delta": delta, 
                    "hedges_g": signed_g,  
                    "p_raw": res[_col(res,"p_val","p-val")].values[0],
                })
    out = pd.DataFrame(rows)
    if not len(out):
        print("NONE"); return out
    rej, p_fdr, _, _ = multipletests(out["p_raw"], alpha=0.05, method="fdr_bh")
    out["p_fdr"] = p_fdr; out["sig_fdr"] = rej
    out["relevant"] = out["sig_fdr"] & (out["hedges_g"].abs() >= 0.2)
    print(f"\nTotal: {len(out)} | FDR significant: {out['sig_fdr'].sum()} "
          f"| relevant (|g|>=0.2): {out['relevant'].sum()}")
    print("\nRELEVANT (|g|>=0.2 e sig. FDR):")
    rel = out[out["relevant"]].sort_values("hedges_g", key=abs, ascending=False)
    print(rel[["model_id","adjective","ethnicity","delta","hedges_g","p_fdr"]].round(3).to_string(index=False)
          if len(rel) else "NONE")
    return out


def block_b_interaction(df):
    d = df[df["ethnicity"] != "baseline"].copy()
    rows = []
    for model in d["model_id"].unique():
        dm = d[d["model_id"] == model]
        for adj in ADJECTIVES:
            sub = dm[["condition","ethnicity",adj]].dropna()
            if sub[adj].nunique() < 2:
                continue
            try:
                aov = pg.anova(data=sub, dv=adj, between=["condition","ethnicity"], detailed=True)
            except Exception as e:
                print(f"  [{model}/{adj}] ANOVA failed: {e}"); continue
            inter = aov[aov["Source"].isin(["condition * ethnicity","ethnicity * condition"])]
            if not len(inter):
                continue
            r = inter.iloc[0]
            rows.append({
                "model_id": model, "adjective": adj,
                "F": r["F"], "p_raw": r[_col(inter,"p-unc","p_unc")],
                "np2": r["np2"], 
            })
    out = pd.DataFrame(rows)
    if not len(out):
        print("NONE"); return out
    rej, p_fdr, _, _ = multipletests(out["p_raw"], alpha=0.05, method="fdr_bh")
    out["p_fdr"] = p_fdr; out["sig_fdr"] = rej
    out["effect_size"] = pd.cut(out["np2"], [-1, ETA_SMALL, ETA_MEDIUM, 1],
                                 labels=["negligible","small","medium+"])
    out["relevant"] = out["sig_fdr"] & (out["np2"] >= ETA_SMALL)
    print(f"\nTested: {len(out)} | sig. FDR: {out['sig_fdr'].sum()} "
          f"| relevant (np2>={ETA_SMALL}): {out['relevant'].sum()}")
    print(out.sort_values("np2", ascending=False)[
        ["model_id","adjective","F","p_fdr","np2","effect_size","relevant"]
    ].round(4).to_string(index=False))
    return out

In [ ]:
import glob
files = glob.glob("output/simplified_ethnicity/experiment_results_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df[ADJECTIVES] = df["scm_raw"].apply(extract_adjectives)  

a = block_a_vs_baseline(df)
b = block_b_interaction(df)
a.to_csv("rq3_vs_baseline.csv", index=False)
b.to_csv("rq3_interaction.csv", index=False)

In [ ]:
plot_forest_block_a(a)
plot_forest_block_a_horizontal(a)
tables_for_block_b(b)

In [ ]:
import pandas as pd
import numpy as np
import pingouin as pg
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 170)

ETA_SMALL = 0.01


def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(names)


def posthoc_within_interactions(df, interaction_csv="rq3_interaction.csv",
                                condition_col="condition", ethnicity_col="ethnicity"):
    rel = pd.read_csv(interaction_csv)
    rel = rel[rel["relevant"]]
    if not len(rel):
        print("NONE")
        return pd.DataFrame()

    d = df[df[ethnicity_col] != "baseline"].copy()
    rows = []

    for _, r in rel.iterrows():
        model, adj = r["model_id"], r["adjective"]
        dm = d[(d["model_id"] == model)]
        if adj not in dm.columns:
            continue

        for cond, g in dm.groupby(condition_col):
            sub = g[[ethnicity_col, adj]].dropna()
            counts = sub.groupby(ethnicity_col)[adj].agg(["count", "nunique"])
            valid_eth = counts[(counts["count"] >= 10) & (counts["nunique"] >= 2)].index
            sub = sub[sub[ethnicity_col].isin(valid_eth)]
            if sub[ethnicity_col].nunique() < 2:
                continue
            try:
                gh = pg.pairwise_gameshowell(data=sub, dv=adj, between=ethnicity_col)
            except Exception:
                continue
            for _, row in gh.iterrows():
                rows.append({
                    "model_id": model, "adjective": adj, "condition": cond,
                    "ethnicity_A": row["A"], "ethnicity_B": row["B"],
                    "mean_A": row["mean_A"], "mean_B": row["mean_B"],
                    "diff": row["diff"], "hedges_g": row["hedges"],
                    "p_raw": row["pval"],
                })

    out = pd.DataFrame(rows)
    if not len(out):
        return out

    rej, pf, _, _ = multipletests(out["p_raw"], alpha=0.05, method="fdr_bh")
    out["p_fdr"] = pf
    out["sig_fdr"] = rej
    out["relevant"] = out["sig_fdr"] & (out["hedges_g"].abs() >= 0.2)
    return out


def flag_condition_specific(posthoc_df):
    rel = posthoc_df[posthoc_df["relevant"]].copy()
    if not len(rel):
        print("NONE")
        return
    for (model, adj, ea, eb), g in rel.groupby(["model_id", "adjective", "ethnicity_A", "ethnicity_B"]):
        if len(g) < 2:
            row = g.iloc[0]
            print(f"\n  [{model}/{adj}] {ea} vs {eb} -> relevant in "
                  f"'{row['condition']}' (g={row['hedges_g']:.2f}, p_fdr={row['p_fdr']:.4f})")
            continue
        g = g.sort_values("hedges_g", key=abs, ascending=False)
        top = g.iloc[0]
        others_mean = g.iloc[1:]["hedges_g"].abs().mean()
        if abs(top["hedges_g"]) > 1.5 * max(others_mean, 0.1):
            print(f"\n  [{model}/{adj}] {ea} vs {eb}: higher in "
                  f"'{top['condition']}' (g={top['hedges_g']:.2f}) compared to others"
                  f" (avg |g| elsewhere={others_mean:.2f})")
    

In [ ]:
files = glob.glob("output/simplified_ethnicity/experiment_results_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

df[ADJECTIVES] = df["scm_raw"].apply(extract_adjectives)   
posthoc = posthoc_within_interactions(df)
if len(posthoc):
    print(f"Total post-hoc: {len(posthoc)} | relevant: {posthoc['relevant'].sum()}")
    r = posthoc[posthoc["relevant"]].sort_values("hedges_g", key=abs, ascending=False)
    print(r[["model_id","adjective","condition","ethnicity_A","ethnicity_B",
            "diff","hedges_g","p_fdr"]].round(3).to_string(index=False))
    flag_condition_specific(posthoc)
    posthoc.to_csv("rq3_posthoc.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

def intensity_by_condition(posthoc_csv="rq3_posthoc.csv"):
    r = pd.read_csv(posthoc_csv)
    r_rel = r[r["relevant"]].copy()

    if not len(r_rel):
        print("NONE"); return


    by_cond = r_rel.groupby("condition")["hedges_g"].agg(
        mean_abs_g=lambda x: x.abs().mean(),
        n_comparisons="count"
    ).sort_values("mean_abs_g", ascending=False)
    print(by_cond.round(3))
    
    by_model_cond = r_rel.groupby(["model_id", "condition"])["hedges_g"].agg(
        mean_abs_g=lambda x: x.abs().mean(),
        n="count"
    ).reset_index()
    pivot = by_model_cond.pivot_table(
        index="model_id", columns="condition", values="mean_abs_g"
    )
    print(pivot.round(2))

    from scipy import stats
    ALL_CONDS = sorted(r["condition"].unique())
    observed = r_rel["condition"].value_counts().reindex(ALL_CONDS, fill_value=0)
    n = observed.sum()
    expected = n / len(ALL_CONDS)
    residuals = (observed - expected) / np.sqrt(expected * (1 - expected / n))

    summary = pd.DataFrame({
        "observed": observed, "expected": round(expected, 1),
        "std_residual": residuals.round(2),
        "pct": (observed / n * 100).round(1)
    }).sort_values("std_residual", ascending=False)
    print(summary)
    chi2, p = stats.chisquare(observed, f_exp=[expected] * len(ALL_CONDS))
    print(f"\nchi2={chi2:.2f}, p={p:.6f}, n={n}")

    return by_cond, pivot, summary, chi2, p

by_cond, pivot, summary, chi2_val, p_val = intensity_by_condition()